In [ ]:
# Configure the repository root once so the same notebook works locally and on Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))

print('Notebook ColdRec cũ: USIM/CGRC đã tắt. Dùng reproduce_paper.ipynb; phép sàng lọc USIM raw riêng là tùy chọn.')


In [ ]:
from src.models.coldrec_adapter import ColdRecDataAdapter, evaluate, eval_subgroups, evaluate_coldrec_embeddings, DUP_GROUPS_PARQUET


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "src/models/coldrec_adapter.py"], cwd=DATA_DIR, check=True)

In [ ]:
import subprocess
from pathlib import Path

COLDREC_COMMIT = "1108192665aca2cb94b3aeb5bfed29850874eb1c"
ALDI_COMMIT = "87f9fe369427402bd989728c4d27b543b2d19cd4"
external_repos = [
    ("ColdRec", "https://github.com/YuanchenBei/ColdRec.git", COLDREC_COMMIT),
    ("ALDI", "https://github.com/zfnWong/ALDI.git", ALDI_COMMIT),
]
for name, url, commit in external_repos:
    destination = DATA_DIR / name
    if not (destination / ".git").is_dir():
        subprocess.run(["git", "clone", "-q", url, str(destination)], check=True)
    subprocess.run(["git", "-C", str(destination), "checkout", "-q", commit], check=True)
    actual = subprocess.run(
        ["git", "-C", str(destination), "rev-parse", "HEAD"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    assert actual == commit, f"{name} commit mismatch: {actual}"
    print(f"{name} commit: {actual}")


In [ ]:
import os
import shutil
import pickle
import numpy as np
from src.models.path_utils import link_or_copy

COLDREC_ROOT = str(DATA_DIR / "ColdRec")
SRC_DATA_DIR = str(DATA_DIR / "ColdRec_Data/MOOCCubeX_raw")
DST_DATA_DIR = os.path.join(COLDREC_ROOT, "data", "MOOCCubeX_raw")
DST_EMB_DIR = os.path.join(COLDREC_ROOT, "emb")

# Đọc ColdRec/data/README.md và kiểm tra cấu trúc thư mục của repo ColdRec
readme_path = os.path.join(COLDREC_ROOT, "data", "README.md")
if os.path.exists(readme_path):
    print("=== NỘI DUNG ColdRec/data/README.md ===")
    with open(readme_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("Không tìm thấy ColdRec/data/README.md, kiểm tra danh sách tệp trong ColdRec:")
    for root, dirs, files in os.walk(COLDREC_ROOT):
        if ".git" in root:
            continue
        level = root.replace(COLDREC_ROOT, "").count(os.sep)
        if level <= 2:
            print(f"{'  ' * level}{os.path.basename(root)}/: {files}")

# Liên kết / sao chép dữ liệu từ ColdRec_Data sang ColdRec/data/ và ColdRec/emb/
os.makedirs(os.path.join(COLDREC_ROOT, "data"), exist_ok=True)
os.makedirs(DST_EMB_DIR, exist_ok=True)

for var in ["raw", "llm", "fused", "llm_3d", "fused_3d"]:
    ds_name = f"MOOCCubeX_{var}"
    src_dir = str(DATA_DIR / f"ColdRec_Data/{ds_name}")
    dst_dir = os.path.join(COLDREC_ROOT, "data", ds_name)
    if os.path.exists(src_dir):
        link_mode = link_or_copy(src_dir, dst_dir)
        print(f"[{link_mode}] {ds_name} -> ColdRec/data/")
        
        # Ánh xạ các tệp embedding sang thư mục ColdRec/emb/ theo mọi quy ước tên mà ColdRec có thể gọi
        for emb_src_name, emb_dst_names in [
            (f"{ds_name}_lightgcn.npy", [f"{ds_name}_lightgcn.npy", f"{ds_name}_warm_emb.npy", "warm_emb.npy"]),
            ("item_content.npy", [f"{ds_name}_item_content.npy", f"{ds_name}_content.npy"]),
        ]:
            s_path = os.path.join(src_dir, emb_src_name)
            if os.path.exists(s_path):
                for d_name in emb_dst_names:
                    d_path = os.path.join(DST_EMB_DIR, d_name)
                    if os.path.islink(d_path) or os.path.exists(d_path):
                        os.remove(d_path)
                    link_or_copy(s_path, d_path)

# Kiểm tra sự tồn tại và kích thước của các tệp bắt buộc
required_files = [
    os.path.join(DST_DATA_DIR, "warm_train.csv"),
    os.path.join(DST_DATA_DIR, "cold_item_val.csv"),
    os.path.join(DST_DATA_DIR, "cold_item_test.csv"),
    os.path.join(DST_DATA_DIR, "para_dict.pkl"),
    os.path.join(DST_DATA_DIR, "item_content.npy"),
    os.path.join(DST_DATA_DIR, "warm_emb.npy"),
    os.path.join(DST_EMB_DIR, "MOOCCubeX_raw_lightgcn.npy"),
]

print("\n=== KIỂM TRA VỊ TRÍ VÀ KÍCH THƯỚC TỆP CHO COLDREC ===")
for p in required_files:
    exists = os.path.exists(p)
    info = ""
    if exists and p.endswith(".npy"):
        arr = np.load(p, mmap_mode="r")
        info = f" | shape={arr.shape}, dtype={arr.dtype}"
    elif exists and p.endswith(".pkl"):
        d = pickle.load(open(p, "rb"))
        info = f" | keys={list(d.keys())}"
    print(f"[{'OK' if exists else 'MISSING'}] {p}{info}")

In [ ]:
import os
import re
import numpy as np
import torch
from src.models.path_utils import link_or_copy

COLDREC_ROOT = str(DATA_DIR / "ColdRec")
COLDREC_DATA_ROOT = str(DATA_DIR / "ColdRec_Data")
DST_EMB_DIR = os.path.join(COLDREC_ROOT, "emb")
init_path = os.path.join(COLDREC_ROOT, "model", "__init__.py")

os.makedirs(DST_EMB_DIR, exist_ok=True)

# ÁNH XẠ TỆP DỮ LIỆU VÀO cold_item/ VÀ BACKBONE (.npy + .pt) SANG ColdRec/emb/
for var in ["raw", "llm", "fused", "llm_3d", "fused_3d"]:
    ds_name = f"MOOCCubeX_{var}"
    ds_dir = os.path.join(COLDREC_DATA_ROOT, ds_name)

    if not os.path.isdir(ds_dir):
        print(f"[SKIP] Không tìm thấy thư mục: {ds_dir}")
        continue

    cold_item_dir = os.path.join(ds_dir, "cold_item")
    os.makedirs(cold_item_dir, exist_ok=True)

    # Liên kết tất cả tệp trực tiếp trong ds_dir vào cold_item/ (nếu chưa có)
    for fname in os.listdir(ds_dir):
        src_file = os.path.join(ds_dir, fname)
        dst_file = os.path.join(cold_item_dir, fname)
        if not os.path.isfile(src_file):
            continue
        link_or_copy(src_file, dst_file)

    print(f"[OK] Đã ánh xạ dữ liệu: {ds_name}/cold_item/")

    # Ánh xạ LightGCN backbone dạng .npy sang ColdRec/emb/
    lgcn_src = os.path.join(ds_dir, f"{ds_name}_lightgcn.npy")
    if os.path.isfile(lgcn_src):
        for candidate_name in [
            f"{ds_name}_LightGCN.npy",
            f"{ds_name}_lightgcn.npy",
            f"{ds_name}_MF.npy",
            f"{ds_name}_cold_item_LightGCN.npy",
            f"{ds_name}_cold_item_MF.npy",
            "LightGCN.npy",
            "MF.npy",
        ]:
            dst_emb = os.path.join(DST_EMB_DIR, candidate_name)
            if os.path.islink(dst_emb) or os.path.exists(dst_emb):
                os.remove(dst_emb)
            link_or_copy(lgcn_src, dst_emb)

    # Tạo các tệp .pt cho --backbone (./emb/..._{backbone}_*.pt) của ColdRec
    gu_src = os.path.join(ds_dir, "g_u.npy")
    gi_src = os.path.join(ds_dir, "g_i.npy")
    if os.path.isfile(gu_src) and os.path.isfile(gi_src):
        gu_t = torch.from_numpy(np.load(gu_src).astype(np.float32))
        gi_t = torch.from_numpy(np.load(gi_src).astype(np.float32))
        concat_t = torch.cat([gu_t, gi_t], dim=0)

        u_pt = os.path.join(ds_dir, "g_u_tensor.pt")
        i_pt = os.path.join(ds_dir, "g_i_tensor.pt")
        c_pt = os.path.join(ds_dir, "g_all_tensor.pt")
        torch.save(gu_t, u_pt)
        torch.save(gi_t, i_pt)
        torch.save(concat_t, c_pt)

        for bb in ["LightGCN", "lightgcn", "MF", "mf"]:
            for prefix in [f"{ds_name}_cold_item_{bb}", f"{ds_name}_{bb}", bb]:
                for suffix, src_pt in [
                    ("_user_emb.pt", u_pt),
                    ("_item_emb.pt", i_pt),
                    ("_user.pt", u_pt),
                    ("_item.pt", i_pt),
                    ("_emb.pt", c_pt),
                    (".pt", c_pt),
                ]:
                    dst_pt = os.path.join(DST_EMB_DIR, f"{prefix}{suffix}")
                    if os.path.islink(dst_pt) or os.path.exists(dst_pt):
                        os.remove(dst_pt)
                    link_or_copy(src_pt, dst_pt)

        print(f"[OK] Đã ánh xạ embedding (.npy & .pt) cho: {ds_name}")

In [ ]:
import glob
import os
from src.models.coldrec_adapter import ColdRecDataAdapter, evaluate, evaluate_coldrec_embeddings
import numpy as np
import torch

DEFAULT_DEV = "cuda:0" if torch.cuda.is_available() else "cpu"


def load_coldrec_model_embeddings(
    model_name,
    dataset_name="MOOCCubeX_raw",
    cold_object="item",
    emb_dir=str(DATA_DIR / "ColdRec/emb"),
    n_users=98220,
    n_items=3769,
):
    """Tự động tìm và đọc (u_emb, i_emb) do ColdRec lưu trong ./emb/ sau khi chạy main.py."""
    # Tìm cặp tệp user_emb / item_emb (.pt hoặc .npy)
    u_candidates = [
        os.path.join(emb_dir, f"{dataset_name}_cold_{cold_object}_{model_name}_user_emb.pt"),
        os.path.join(emb_dir, f"{dataset_name}_{model_name}_user_emb.pt"),
        os.path.join(emb_dir, f"{dataset_name}_cold_{cold_object}_{model_name}_user_emb.npy",),
        os.path.join(emb_dir, f"{dataset_name}_{model_name}_user_emb.npy"),
    ]
    i_candidates = [
        os.path.join(emb_dir, f"{dataset_name}_cold_{cold_object}_{model_name}_item_emb.pt"),
        os.path.join(emb_dir, f"{dataset_name}_{model_name}_item_emb.pt"),
        os.path.join(emb_dir, f"{dataset_name}_cold_{cold_object}_{model_name}_item_emb.npy",),
        os.path.join(emb_dir, f"{dataset_name}_{model_name}_item_emb.npy"),
    ]

    def _load_arr(path):
        if path.endswith(".pt"):
            obj = torch.load(path, map_location="cpu", weights_only=False)
        return (
            obj.detach().cpu().numpy()
            if isinstance(obj, torch.Tensor)
            else np.asarray(obj)
        )
        return np.load(path)

    for up, ip in zip(u_candidates, i_candidates):
        if os.path.exists(up) and os.path.exists(ip):
            return _load_arr(up), _load_arr(ip)

    # Tìm tệp gộp (shape = [n_users + n_items, 64] hoặc dict {'user_emb', 'item_emb'})
    pattern_files = sorted(glob.glob(os.path.join(emb_dir, f"*{model_name}*")),key=os.path.getmtime,reverse=True,)
    for p in pattern_files:
        if p.endswith((".pt", ".npy")):
            data = _load_arr(p)
        if (
            isinstance(data, dict)
            and "user_emb" in data
            and "item_emb" in data
        ):
            u_a = (
                data["user_emb"].detach().cpu().numpy()
                if isinstance(data["user_emb"], torch.Tensor)
                else np.asarray(data["user_emb"])
            )
            i_a = (
                data["item_emb"].detach().cpu().numpy()
                if isinstance(data["item_emb"], torch.Tensor)
                else np.asarray(data["item_emb"])
            )
            return u_a, i_a
        if isinstance(data, np.ndarray) and data.shape[0] == n_users + n_items:
            return data[:n_users], data[n_users:]

    raise FileNotFoundError(
        f"Không tìm thấy embedding đã lưu của {model_name} trong {emb_dir}."
        f" Các tệp hiện có: {os.listdir(emb_dir)}"
    )


@torch.no_grad()
def evaluate_coldrec_val_only(
    adapter,
    u_emb,
    i_emb,
    model_name,
    variant_name="raw",
    seed=42,
    device=DEFAULT_DEV,
):
    """Chấm điểm NDCG@20 và Recall@20 trên tập COLD-VAL (251 khóa cold-val, 2.092 học viên)
    để chọn mô hình giữa USIM và CGRC theo đúng quy tắc (KHÔNG xem tập
    test).
    """
    u_t = torch.as_tensor(u_emb, dtype=torch.float32, device=device)
    i_t = torch.as_tensor(i_emb, dtype=torch.float32, device=device)

    # Kiểm tra điều kiện bắt buộc: xuất đủ điểm cho 98.220 học viên x 3.769 khóa
    assert u_t.shape == (
        98220,
        64,
    ), f"Sai kích thước u_emb: {u_t.shape}, yêu cầu (98220, 64)"
    assert i_t.shape == (
        3769,
        64,
    ), f"Sai kích thước i_emb: {i_t.shape}, yêu cầu (3769, 64)"
    assert (
        not torch.isnan(u_t).any() and not torch.isnan(i_t).any()
    ), "Có giá trị NaN trong embedding!"

    score_fn = lambda ub: (u_t[ub.to(device)] @ i_t.t()).cpu()

    # Đánh giá trên 251 khóa cold-val (cand_val_a), che lịch sử train (hist_tr)
    res_val = evaluate(
        score_fn,
        adapter.val_target_dict,
        cand=adapter.cand_val_a,
        mask=adapter.hist_tr,
        ks=(5, 10, 20),
        tie_seed=seed,
    )
    val_ndcg20 = float(np.mean(res_val["NDCG@20"]))
    val_recall20 = float(np.mean(res_val["Recall@20"]))

    # Lưu embedding của cấu hình mặc định để truy vết
    rep_dir = os.path.join(
        adapter.output_dir, f"results_{model_name}", "default_val_reps"
    )
    os.makedirs(rep_dir, exist_ok=True)
    np.save(
        os.path.join(rep_dir, f"u_e_{variant_name}_seed{seed}.npy"),
        u_t.cpu().numpy(),
    )
    np.save(
        os.path.join(rep_dir, f"v_e_{variant_name}_seed{seed}.npy"),
        i_t.cpu().numpy(),
    )

    print(
        f"[{model_name.upper()} - DEFAULT (seed={seed}, {variant_name})] "
        f"Kích thước: {u_t.shape[0]} HV x {i_t.shape[0]} khóa | "
        f"Cold-Val NDCG@20 = {val_ndcg20:.4f} | Cold-Val Recall@20 ="
        f" {val_recall20:.4f}"
    )
    return val_ndcg20, val_recall20

In [ ]:
print(
    "Bỏ qua USIM và CGRC trong notebook ColdRec cũ. "
    "USIM chỉ được sàng lọc raw trong coldrec-adapter-usim-raw.ipynb; "
    "CGRC không thuộc quy trình tái lập."
)

In [ ]:
from src.models.run_aldi import ALDI, tune_and_run_aldi


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "src/models/run_aldi.py"], cwd=DATA_DIR, check=True)